# ScholarLens 04 - Results tables and live demo
Owner: Member 4. Builds all comparison tables/figures and launches the Gradio demo with a public link for the presentation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sajithaucm/Scholarlens_Project/blob/main/notebooks/04_results_and_demo.ipynb)

## 0. Setup (run every time you open this notebook)
1. **Runtime -> Change runtime type -> T4 GPU** (Hardware accelerator).
2. Replace `sajithaucm` below with your team's GitHub account (once, then commit).
3. Checkpoints are saved to **Google Drive** so they survive a disconnect; small result files are saved in the repo's `results/` folder and pushed to GitHub at the end.

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/sajithaucm/Scholarlens_Project.git"
if not os.path.exists("/content/Scholarlens_Project"):
    !git clone $REPO /content/Scholarlens_Project
%cd /content/Scholarlens_Project
!git pull -q
!pip install -q -r requirements.txt

# Checkpoints -> Google Drive (big files never go to GitHub)
os.environ["SCHOLARLENS_OUTPUT_DIR"] = "/content/drive/MyDrive/Scholarlens_Project/outputs"
os.makedirs(os.environ["SCHOLARLENS_OUTPUT_DIR"], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze > results/environment.txt   # exact library versions, for reproducibility

## 1. Comparison tables and figures

In [ ]:
!git pull -q
!python -m src.compare
from IPython.display import Markdown, Image, display
display(Markdown(open('results/RESULTS.md').read()))

In [ ]:
import glob
for f in sorted(glob.glob('results/fig_*.png')):
    print(f); display(Image(f))

## 2. Launch the demo
The app loads the checkpoints named in `configs/config.yaml -> demo` from your Drive output folder. Make sure they are your **best** model of each pair (edit and commit the config if not). Open the `*.gradio.live` link it prints and relaunch it shortly before presenting.

In [ ]:
!python -m app.app --share

## Save results to GitHub
Results (`results/*.json`, figures, sample predictions) are small and must be committed.
Create a GitHub **fine-grained personal access token** (repo contents: read & write) and paste it when asked. It is not stored in the notebook.
Alternative: download the `results/` folder (Files panel) and commit it from VS Code.

In [ ]:
from getpass import getpass
name, email = input("Your name: "), input("Your GitHub email: ")
token = getpass("GitHub token: ")
!git config user.name "$name" && git config user.email "$email"
!git add results/
!git commit -m "Add results from Colab run" || echo "nothing to commit"
!git push https://$token@github.com/sajithaucm/Scholarlens_Project.git HEAD:main
del token